# Cross-trait and multi-polytranscriptional risk score analysis of PD in AMP-PD: Penalised regression multi-PTS models

**Project**: Cross-trait and multi-polytranscriptomic score analysis of Parkinson's disease identifies novel associations and improves prediction

**Date last updated**: July 2026 

 ## Initial set-up 

### Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os
import sys

# Bring in Pandas for Dataframe functionality
import pandas as pd
from functools import reduce

# Bring some visualization functionality 
import seaborn as sns  

# numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Enable interaction with the FireCloud API
from firecloud import api as fapi

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

### Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

# Utility routines for reading files from Google Cloud Storage
def gcs_read_file(path):
    """Return the contents of a file in GCS"""
    contents = !gsutil -u {BILLING_PROJECT_ID} cat {path}
    return '\n'.join(contents)
    
def gcs_read_csv(path, sep=None):
    """Return a DataFrame from the contents of a delimited file in GCS"""
    return pd.read_csv(StringIO(gcs_read_file(path)), sep=sep, engine='python')

# Utility routine for displaying a message and link to Cloud Console
def link_to_cloud_console_gcs(description, link_text, gcs_path):
    url = '{}?{}'.format(
        os.path.join('https://console.cloud.google.com/storage/browser',
                     gcs_path.replace("gs://","")),
        urllib.parse.urlencode({'userProject': BILLING_PROJECT_ID}))

    display_html_link(description, link_text, url)

### Set paths

In [ ]:
# Set up billing project and data path variables
BILLING_PROJECT_ID = os.environ['GOOGLE_PROJECT']
WORKSPACE_NAMESPACE = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE_NAME = os.environ['WORKSPACE_NAME']
WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']
WORKSPACE_ATTRIBUTES = fapi.get_workspace(WORKSPACE_NAMESPACE, WORKSPACE_NAME).json().get('workspace',{}).get('attributes',{})

## Print the information to check we are in the proper release and billing 
## This will be different for you, the user, depending on the billing project your workspace is on
print('Billing and Workspace')
print(f'Workspace Name @ `WORKSPACE_NAME`: {WORKSPACE_NAME}')
print(f'Billing Project @ `BILLING_PROJECT_ID`: {BILLING_PROJECT_ID}')
print(f'Workspace Bucket, where you can upload and download data @ `WORKSPACE_BUCKET`: {WORKSPACE_BUCKET}')
print('')

## AMP-PD v4.0
# Explicitly define release v4.0 path 
AMP_RELEASE_CASE_CONTROL_PATH = 'gs://path/removed'
AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'
AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'

#rnaseq_WB-RWTS-VHBS_samples.csv
#rnaseq_WB-RWTS_samples.csv


print('AMP-PD v4.0')
print(f'Path to AMP-PD v4.0 case/control data: {AMP_RELEASE_CASE_CONTROL_PATH}')
print(f'Path to AMP-PD v4.0 PPMI and PDBP RNA Data: {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}')
print(f'Path to AMP-PD v4.0 HBS Data: {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}')

### Make some directories and load R

In [ ]:
!ls /home/jupyter/multiTRS/

# Make a directories for the scorefiles
!mkdir -p /home/jupyter/multiTRS/multi_score_output
!ls /home/jupyter/multiTRS/
!mkdir -p /home/jupyter/multiTRS/multi_score_output/TWAS_SMR_FDR_models
!ls /home/jupyter/multiTRS/multi_score_output/

### Load R and install glmnet

In [ ]:
!pip install rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R

install.packages("glmnet")
install.packages("pROC")
install.packages("caret")
remotes::install_github("hongooi73/glmnetUtils")

## Residualise scores for surrogate variables

There are different numbers of surrogate variables per cohort. 

So the multi-TRS models can be applied to each dataset following training we residualise the TRS scores within each for the surrogate variables.  

For HBS there were no significant SVs, we can jsut copy the original scores and rename with _resid_ so they match later

In [ ]:
%%R

# Load packages
library(data.table)
library(dplyr)
library(stringr)

HBS_scores <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores.txt")

id_col <- HBS_scores %>% select(participant_id)
print(paste0("The ID column is ", names(id_col)))

score_cols <- setdiff(names(HBS_scores), "participant_id")

setnames(
  HBS_scores,
  old = score_cols,
  new = paste0(score_cols, "_resid")
)

print(HBS_scores)

write.table(HBS_scores, "/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

In [ ]:
%%R

# Load packages
library(data.table)
library(dplyr)
library(stringr)

# List of cohorts
cohorts <- c("PDBP", "PPMI")

# Loop over cohorts
for (cohort in cohorts) {
  
  cat("\nProcessing cohort:", cohort, "\n")
  
  # ---------------------------
  # 1. Read scores and SV files
  # ---------------------------
  scores_path <- paste0("/home/jupyter/multiTRS/scores/", cohort, "_TWAS_SMR_FDR_scores.txt")
  SV_path <- paste0("/home/jupyter/multiTRS/RNA/clean/", cohort, "_SVs.txt")
  
  if (!file.exists(scores_path)) stop(paste("Scores file not found:", scores_path))
  if (!file.exists(SV_path)) stop(paste("SV file not found:", SV_path))
  
  scores <- fread(scores_path) %>% as.data.frame()
    
  print(head(scores[,2]))
    
  SV <- fread(SV_path) %>% as.data.frame()
  
  # ---------------------------
  # 2. Join by ID column
  # ---------------------------
  # Replace 'ID' with the actual common ID column name
  combined_scores_SV <- inner_join(SV, scores, by = "participant_id")
  
  # ---------------------------
  # 3. Identify columns
  # ---------------------------
  id_col <- combined_scores_SV %>% select(participant_id)
  print(paste0("The ID column is ",names(id_col)))
    
  SV_cols <- combined_scores_SV %>% select(starts_with("SV"))
  print(paste0("The SV columns are ",names(SV_cols)))
    
  score_cols <- combined_scores_SV %>% 
  select(-all_of(names(id_col)), -all_of(names(SV_cols)))
  #print(paste0("The score columns are ",names(score_cols)))
  
  # ---------------------------
  # 4. Residualise each score for SVs
  # ---------------------------
  resid_scores <- lapply(score_cols, function(y) {
    resid(lm(y ~ ., data = SV_cols))
    #print(summary(lm(y ~ ., data = SV_cols)))
  })
  
  resid_scores <- as.data.frame(resid_scores)
  names(resid_scores) <- paste0(names(score_cols), "_resid")
  
  # ---------------------------
  # 5. Combine ID + residualised scores
  # ---------------------------
  df_resid <- cbind(id_col, resid_scores)
  
  # Print first few rows for sanity check
  print(head(df_resid[,2]))
  
  # ---------------------------
  # 6. Save file
  # ---------------------------
  outfile <- paste0("/home/jupyter/multiTRS/scores/",cohort,"_TWAS_SMR_FDR_scores_resid.txt")
  print(paste0("Saving residualised file: ",outfile))
  write.table(df_resid, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
  
}


## Multi-PTS

### LASSO

#### SMR-multi scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using SMR-multi scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("SMR"), -contains("single_SNP"))
  
  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "SMR-multi",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("SMR"), -contains("single_SNP"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing SMR-multi model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.04566523,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

#### SMR-single-SNP scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using SMR-multi scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))
  
  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "SMR single SNP",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing SMR-single-SNP model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP <- predict(null_model, newdata = combined_data,      type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP <- auc(roc(combined_data$case_control_other_at_baseline,      null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("PD_PGS_resid", "sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.03190276,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

#### FUSION scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using FUSION scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_FUSION_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("FUSION"))
  
  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "FUSION",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("FUSION"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing FUSION model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.0536878,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_FUSION_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_FUSION_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

#### All scores (SMR-multi, SMR-single-SNP, FUSION)

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates LASSO multi-TRS models using all scores... Running LASSO models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_all_scores_LASSO_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {
  
  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))
  
  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  
  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")
  
  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + age_at_baseline,
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")
  
  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("threshold", "sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])
  
  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)
  
  cat("Number of predictors:", ncol(X), "\n")
  
  # Define penalty factors: unpenalized for sex and age
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
  
  # Outer 10-fold CV
  outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)
  
  # Initialize vectors for storing performance
  outer_auc <- c()
  lambda_vals <- c()
  outer_sensitivity <- c()
  outer_specificity <- c()
  outer_accuracy <- c()
  
  for (i in seq_along(outer_folds)) {
    cat("Outer fold:", i, "\n")
    
    train_idx <- outer_folds[[i]]
    test_idx <- setdiff(seq_along(y), train_idx)
    
    X_train <- X[train_idx, , drop = FALSE]
    y_train <- y[train_idx]
    X_test <- X[test_idx, , drop = FALSE]
    y_test <- y[test_idx]
    
    # Standardise variables (exclude sex)
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
    # Inner CV for LASSO (alpha = 1)
    cvfit <- cv.glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      nfolds = 5,
      type.measure = "auc",
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    best_lambda <- cvfit$lambda.1se
    lambda_vals[i] <- best_lambda
    
    # Refit model on full training set with best lambda
    final_model <- glmnet(
      x = X_train_scaled,
      y = y_train,
      family = "binomial",
      alpha = 1,
      lambda = best_lambda,
      penalty.factor = penalty,
      standardize = FALSE
    )
    
    # Predict on outer test set
    probs <- predict(final_model, newx = X_test_scaled, type = "response")
    roc_obj_outer <- roc(y_test, as.numeric(probs))
    auc_val <- auc(roc_obj_outer)
    
    model_coords <- coords(roc_obj_outer, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
    
    outer_auc[i] <- auc_val
    outer_sensitivity[i] <- as.numeric(model_coords["sensitivity"])
    outer_specificity[i] <- as.numeric(model_coords["specificity"])
    outer_accuracy[i] <- as.numeric(model_coords["accuracy"])
  }
  
  # Median lambda across outer folds
  lambda_final <- median(lambda_vals)
    
  cat("AUC for each fold in ", TRS_cohort, ":", outer_auc, "\n")
  
  cat("Median lambda for cohort", TRS_cohort, ":", lambda_final, "\n")
  cat("Mean outer AUC:", mean(outer_auc), "SD:", sd(outer_auc), "\n")

  
  # Store performance results
  performance_results <- data.table(
    cohort = TRS_cohort,
    scores = "all_scores",
    model = "LASSO",
    covariate_treatment = "unpenalised",
    median_lambda = lambda_final,
    median_alpha = "Not applicable",
    AUC_null = auc_null,
    AUC_full_mean_cv = mean(outer_auc),
    AUC_full_sd_cv = sd(outer_auc),
    AUC_diff = mean(outer_auc) - auc_null,
    sens_null = null_sensitivity,
    sens_full_mean_cv = mean(outer_sensitivity),
    sens_full_sd_cv = sd(outer_sensitivity),
    sens_diff = mean(outer_sensitivity) - null_sensitivity,
    spec_null = null_specificity,
    spec_full_mean_cv = mean(outer_specificity),
    spec_full_sd_cv = sd(outer_specificity),
    spec_diff = mean(outer_specificity) - null_specificity,
    acc_null = null_accuracy,
    acc_full_mean_cv = mean(outer_accuracy),
    acc_full_sd_cv = sd(outer_accuracy),
    acc_diff = mean(outer_accuracy) - null_accuracy
  )
  
  all_performance_results <- rbind(all_performance_results, performance_results)
  
}


print(all_performance_results)

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing FUSION model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 1,
  lambda = 0.04599794,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_all_scores_LASSO_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")


# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_all_scores_LASSO_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### ENET

#### SMR-multi scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using SMR-multi scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("SMR"), -contains("single_SNP"))

  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "SMR-multi",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("SMR"), -contains("single_SNP"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing SMR-multi ENET model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.3,
  lambda = 0.1598872,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

#### SMR single-SNP scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using SMR-multi scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))

  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "SMR single SNP",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("single_SNP"),-contains("Hip_Fracture"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing SMR-single-SNP ENET model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.1,
  lambda = 0.2414718,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

#### FUSION scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using FUSION scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_SMR_FUSION_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(participant_id, contains("FUSION"))

  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "FUSION",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(participant_id, contains("fusion"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing FUSION ENET model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0,
  lambda = 2.540931,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_FUSION_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,          y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_FUSION_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

#### All scores

##### Nested-CV

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(glmnetUtils)
library(caret)
library(pROC)

set.seed(1)

# Create a data.table for all performance results
all_performance_results <- data.table()

print("This script calculates ENET multi-TRS models using all scores... Running ENET models")

outfile <- "/home/jupyter/multiTRS/results/multi_TRS_all_scores_ENET_nestedcv.txt"

# Read in the clinical data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>%
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)

# List of cohorts to loop over
TRS_cohorts <- c("PDBP")

# Loop over cohorts
for (TRS_cohort in TRS_cohorts) {

  # Read TRS scores
  TRS_path <- paste0("/home/jupyter/multiTRS/scores/", TRS_cohort, "_TWAS_SMR_FDR_scores_resid.txt")
  TRS <- fread(TRS_path) %>% select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))

  combined_data <- clinical %>%
    inner_join(TRS, by = "participant_id")

  cat("Rows in combined data for", TRS_cohort, ":", nrow(combined_data), "\n")

  # Null model with age + sex
  null_model <- glm(case_control_other_at_baseline ~ sex + scale(age_at_baseline),
                    data = combined_data,
                    family = binomial)
  probs_null <- predict(null_model, type = "response")
  roc_obj_null <- roc(combined_data$case_control_other_at_baseline, probs_null)
  auc_null <- auc(roc_obj_null)
  cat("AUC of null model:", auc_null, "\n")

  null_coords <- coords(roc_obj_null, x = "best", best.method = "closest.topleft",
                        ret = c("sensitivity", "specificity", "accuracy"))
  null_sensitivity <- as.numeric(null_coords["sensitivity"])
  null_specificity <- as.numeric(null_coords["specificity"])
  null_accuracy <- as.numeric(null_coords["accuracy"])

  # Prepare predictor matrix X and outcome y
  X <- combined_data %>%
    select(-participant_id, -case_control_other_at_baseline) %>%
    as.matrix()
  y <- as.factor(combined_data$case_control_other_at_baseline)

  cat("Number of predictors:", ncol(X), "\n")

  # Define penalty factors: unpenalized for sex, age and PD_PGS_resid
  penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)

# Create five outer folds
outer_folds <- createFolds(y, k = 5, returnTrain = TRUE)

# Store auc of outer folders
outer_auc <- c()
lambda_vals <- c()
alpha_vals  <- c()
outer_sensitivity <- c()
outer_specificity <- c()
outer_accuracy <- c()

for (i in 1:length(outer_folds)) {
  cat("Outer fold:", i, "\n")
  
  # Train/test split for this outer fold
  train_idx <- outer_folds[[i]]
  test_idx  <- setdiff(seq_along(y), train_idx)
  
  X_train <- X[train_idx, ]
  y_train <- y[train_idx]
  X_test  <- X[test_idx, ]
  y_test  <- y[test_idx]
  
  # Standardize inside training only
    vars_to_scale <- setdiff(colnames(X), "sex")
    scaler <- preProcess(X_train[, vars_to_scale, drop = FALSE], method = c("center", "scale"))
    
    X_train_scaled <- X_train
    X_train_scaled[, vars_to_scale] <- predict(scaler, X_train[, vars_to_scale, drop = FALSE])
    
    X_test_scaled <- X_test
    X_test_scaled[, vars_to_scale] <- predict(scaler, X_test[, vars_to_scale, drop = FALSE])
    
 # Specify alpha list 
  alphalist <- seq(0,1,by=0.1)
  
  # Inner CV with cv.glmnet (5-fold)
  cvfit <- glmnetUtils::cva.glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = alphalist,
    nfolds = 5,
    type.measure = "auc",
    penalty.factor = penalty,
    standardize = FALSE
  )
  

    
  # Select best alpha and lambda
  lambda_1se <- sapply(cvfit$modlist, `[[`, "lambda.1se")
  error <- sapply(cvfit$modlist, function(mod) {
    idx <- which(mod$lambda == mod$lambda.1se)
    mod$cvm[idx]
  })
  best <- which.max(error)
  best_alpha  <- cvfit$alpha[best]
  best_lambda <- lambda_1se[best]
                  

  # Best lambda chosen inside
  lambda_vals[i] <- best_lambda
  alpha_vals[i]  <- best_alpha
    
  
  # Refit on full training set with best lambda
  final_model <- glmnet(
    x = X_train_scaled,
    y = y_train,
    family = "binomial",
    alpha = best_alpha,
    lambda = best_lambda,
    penalty.factor = penalty,
    standardize = FALSE
  )
  
  # Predict probabilities on outer test set
  probs <- predict(final_model, newx = X_test_scaled, type = "response")
  
# Compute ROC curve
  roc_obj_outer <- roc(y_test, as.numeric(probs))

# Compute AUC
  auc_val <- auc(roc_obj_outer)

# Get sens, spec, accuracy using topleft
model_coords <- coords(
  roc_obj_outer,
  x = "best",
  best.method = "closest.topleft",
  ret = c("threshold", "sensitivity", "specificity", "accuracy")
)

model_sensitivity <- as.numeric(model_coords["sensitivity"])
model_specificity <- as.numeric(model_coords["specificity"])
model_accuracy <- as.numeric(model_coords["accuracy"])
    
  outer_auc[i] <- auc_val
  outer_sensitivity[i] <- model_sensitivity
  outer_specificity[i] <- model_specificity
  outer_accuracy[i] <- model_accuracy
}

alpha_vals <- as.numeric(alpha_vals)
lambda_vals <- as.numeric(lambda_vals)

cat("Alpha per outer fold:\n")
print(alpha_vals)

cat("Lambda per outer fold:\n")
print(lambda_vals)
                  
lambda_final <- median(lambda_vals)
alpha_final <- median(alpha_vals)


cat("AUC per outer fold:\n")
print(outer_auc)
cat("Mean AUC across outer folds:", mean(outer_auc), "\n")
cat("SD of the AUC across outer folds:", sd(outer_auc), "\n")

cat("Median alpha:\n")
print(alpha_final)
cat("Median lambda:\n")
print(lambda_final)


                  
performance_results <- data.table(cohort = TRS_cohort,
                                  scores = "all_scores",
                                  model = "ENET",
                                  covariate_treatment = "unpenalised",
                                  median_lambda = lambda_final,
                                  median_alpha = alpha_final,
                                  AUC_null = auc_null,
                                  AUC_full_mean_cv =  mean(outer_auc),
                                  AUC_full_sd_cv = sd(outer_auc),
                                  AUC_diff = mean(outer_auc) - auc_null,
                                  sens_null = null_sensitivity,
                                  sens_full_mean_cv = mean(outer_sensitivity),
                                  sens_full_sd_cv = sd(outer_sensitivity),
                                  sens_diff = mean(outer_sensitivity) - null_sensitivity,
                                  spec_null = null_specificity,
                                  spec_full_mean_cv = mean(outer_specificity),
                                  spec_full_sd_cv = sd(outer_specificity),
                                  spec_diff = mean(outer_specificity) - null_specificity,
                                  acc_null = null_accuracy,
                                  acc_full_mean_cv = mean(outer_accuracy),
                                  acc_full_sd_cv = sd(outer_accuracy),
                                  acc_diff = mean(outer_accuracy) - null_accuracy)
    

all_performance_results <- rbind(all_performance_results,performance_results)


}
                  
print(all_performance_results)
                  

write.table(all_performance_results, outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

##### Test in PPMI and HBS

In [ ]:
%%R

library(data.table)
library(dplyr)
library(glmnet)
library(caret)
library(pROC)

set.seed(1)

# Read in the data
clinical <- fread("/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt") %>% 
  select(participant_id, case_control_other_at_baseline, sex, age_at_baseline)


# Read TRS scores - select columns based on PDBP first
TRS_PDBP <- fread("/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(-contains("Hip_fracture_EUR_2022_FDR_SMR_single_SNP"))

TRS_PPMI <- fread("/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

TRS_HBS  <- fread("/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt") %>% 
  select(all_of(colnames(TRS_PDBP)))

# -----------------------------------------------
# Create combined datasets
# -----------------------------------------------

# PDBP (training)
combined_data <- clinical %>%
  inner_join(TRS_PDBP, by = "participant_id")

# PPMI (test)
combined_data_PPMI <- clinical %>%
  inner_join(TRS_PPMI, by = "participant_id")


# HBS (test)
combined_data_HBS <- clinical %>%
  inner_join(TRS_HBS, by = "participant_id")

cat("Testing all scores ENET model...\n")
cat("Rows in combined data - PDBP:", nrow(combined_data), "\n")
cat("Rows in combined data - PPMI:", nrow(combined_data_PPMI), "\n")
cat("Rows in combined data - HBS:", nrow(combined_data_HBS), "\n")

# -----------------------------------------------
# Null model (age + sex), fit on PDBP, applied to all
# -----------------------------------------------

null_model <- glm(case_control_other_at_baseline ~ scale(age_at_baseline) + sex,
                  data = combined_data,
                  family = binomial)

# Null model predictions
null_probs_PDBP     <- predict(null_model, newdata = combined_data,          type = "response")
null_probs_PPMI <- predict(null_model, newdata = combined_data_PPMI, type = "response")
null_probs_HBS  <- predict(null_model, newdata = combined_data_HBS,  type = "response")

auc_null_PDBP     <- auc(roc(combined_data$case_control_other_at_baseline,          null_probs_PDBP))
auc_null_PPMI <- auc(roc(combined_data_PPMI$case_control_other_at_baseline, null_probs_PPMI))
auc_null_HBS  <- auc(roc(combined_data_HBS$case_control_other_at_baseline,  null_probs_HBS))

cat("Null model AUCs:\n")
cat("  PDBP:", auc_null_PDBP, "\n")
cat("  PPMI:", auc_null_PPMI, "\n")
cat("  HBS:", auc_null_HBS, "\n")

# -----------------------------------------------
# Prepare predictor matrices
# -----------------------------------------------

X <- combined_data %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y <- as.factor(combined_data$case_control_other_at_baseline)

X_PPMI <- combined_data_PPMI %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_PPMI <- as.factor(combined_data_PPMI$case_control_other_at_baseline)


X_HBS <- combined_data_HBS %>%
  select(-participant_id, -case_control_other_at_baseline) %>% as.matrix()
y_HBS <- as.factor(combined_data_HBS$case_control_other_at_baseline)

cat("Number of predictors:", ncol(X), "\n")

# -----------------------------------------------
# Scale on PDBP, apply to all
# -----------------------------------------------

penalty <- ifelse(colnames(X) %in% c("sex", "age_at_baseline"), 0, 1)
vars_to_scale <- setdiff(colnames(X), "sex")

scaler_full <- preProcess(X[, vars_to_scale], method = c("center", "scale"))

X_scaled          <- X
X_scaled[, vars_to_scale] <- predict(scaler_full, X[, vars_to_scale])

X_PPMI_scaled <- X_PPMI
X_PPMI_scaled[, vars_to_scale] <- predict(scaler_full, X_PPMI[, vars_to_scale])

X_HBS_scaled  <- X_HBS
X_HBS_scaled[, vars_to_scale]  <- predict(scaler_full, X_HBS[, vars_to_scale])

# -----------------------------------------------
# Fit final LASSO model on PDBP
# -----------------------------------------------

final_model <- glmnet(
  x = X_scaled,
  y = y,
  family = "binomial",
  alpha = 0.4,
  lambda = 0.08403927,
  penalty.factor = penalty,
  standardize = FALSE
)

# Extract and print coefficients
coefs <- coef(final_model)
coef_df <- data.frame(
  feature = rownames(coefs),
  coefficient = as.numeric(coefs)
)

coef_df <- coef_df %>% arrange(desc(coefficient))

print(coef_df)

coef_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_all_scores_ENET_PDBP_full_model_coefficients.txt"
write.table(coef_df, coef_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Coefficients saved to:", coef_outfile, "\n")

# -----------------------------------------------
# Predictions and AUCs
# -----------------------------------------------

get_auc <- function(model, X_new, y_new) {
  probs <- predict(model, newx = X_new, type = "response")
  auc(roc(y_new, as.numeric(probs)))
}

auc_PDBP     <- get_auc(final_model, X_scaled,y)
auc_PPMI <- get_auc(final_model, X_PPMI_scaled, y_PPMI)
auc_HBS  <- get_auc(final_model, X_HBS_scaled,  y_HBS)

cat("Full model AUCs:\n")
cat("  PDBP:", auc_PDBP, "\n")
cat("  PPMI:", auc_PPMI, "\n")
cat("  HBS:", auc_HBS, "\n")

# -----------------------------------------------
# ROC objects (created before CIs and plots)
# -----------------------------------------------

probs_PPMI <- predict(final_model, newx = X_PPMI_scaled, type = "response")
probs_HBS  <- predict(final_model, newx = X_HBS_scaled,  type = "response")

roc_null_PPMI <- roc(y_PPMI, null_probs_PPMI)
roc_full_PPMI <- roc(y_PPMI, as.numeric(probs_PPMI))

roc_null_HBS  <- roc(y_HBS,  null_probs_HBS)
roc_full_HBS  <- roc(y_HBS,  as.numeric(probs_HBS))

# -----------------------------------------------
# CIs for AUCs
# -----------------------------------------------

ci_null_PPMI <- ci.auc(roc_null_PPMI)
ci_null_HBS  <- ci.auc(roc_null_HBS)

ci_full_PPMI <- ci.auc(roc_full_PPMI)
ci_full_HBS  <- ci.auc(roc_full_HBS)

cat("Null model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_null_PPMI, 3), "(95% CI:", round(ci_null_PPMI[1], 3), "-", round(ci_null_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_null_HBS, 3),   "(95% CI:", round(ci_null_HBS[1], 3),  "-", round(ci_null_HBS[3], 3),  ")\n")

cat("Full model AUCs with 95% CI:\n")
cat("  PPMI:", round(auc_PPMI, 3), "(95% CI:", round(ci_full_PPMI[1], 3), "-", round(ci_full_PPMI[3], 3), ")\n")
cat("  HBS:",  round(auc_HBS, 3),   "(95% CI:", round(ci_full_HBS[1], 3),  "-", round(ci_full_HBS[3], 3),  ")\n")

# -----------------------------------------------
# CI bands for plotting (ci.se required for ci.type="shape")
# -----------------------------------------------

ci_se_null_PPMI <- ci.se(roc_null_PPMI, specificities = seq(0, 1, 0.01))
ci_se_full_PPMI <- ci.se(roc_full_PPMI, specificities = seq(0, 1, 0.01))

ci_se_null_HBS  <- ci.se(roc_null_HBS,  specificities = seq(0, 1, 0.01))
ci_se_full_HBS  <- ci.se(roc_full_HBS,  specificities = seq(0, 1, 0.01))

# -----------------------------------------------
# ROC plots with CI shapes
# -----------------------------------------------

plot_roc <- function(roc_null, roc_full, ci_se_null, ci_se_full, ci_null, ci_full, title, full_col) {

  # Plot null model ROC
  plot.roc(roc_null, col = "black", lwd = 2, main = title,
           legacy.axes = TRUE, print.auc = FALSE)

  # Add CI shape for null model
  plot(ci_se_null, type = "shape",
       col = adjustcolor("black", alpha.f = 0.1), border = NA)

  # Add full model ROC
  plot.roc(roc_full, add = TRUE, col = full_col, lwd = 2)

  # Add CI shape for full model
  plot(ci_se_full, type = "shape",
       col = adjustcolor(full_col, alpha.f = 0.15), border = NA)

  # Redraw ROC lines on top of shading
  lines.roc(roc_null, col = "black", lwd = 2)
  lines.roc(roc_full, col = full_col, lwd = 2)

  # Legend with AUC + 95% CI
  legend("bottomright",
         legend = c(
           paste0("Age + Sex (AUC = ", round(auc(roc_null), 3),
                  " [", round(ci_null[1], 3), "\u2013", round(ci_null[3], 3), "])"),
           paste0("Full Model (AUC = ", round(auc(roc_full), 3),
                  " [", round(ci_full[1], 3), "\u2013", round(ci_full[3], 3), "])")
         ),
         col = c("black", full_col), lwd = 2, bty = "n")
}

plot_roc(roc_null_PPMI, roc_full_PPMI, ci_se_null_PPMI, ci_se_full_PPMI, ci_null_PPMI, ci_full_PPMI, "PPMI: Baseline vs Full Model", "goldenrod")
plot_roc(roc_null_HBS,  roc_full_HBS,  ci_se_null_HBS,  ci_se_full_HBS,  ci_null_HBS,  ci_full_HBS,  "HBS: Baseline vs Full Model",  "dodgerblue")

# -----------------------------------------------
# DeLong tests
# -----------------------------------------------

delong_PPMI <- roc.test(roc_null_PPMI, roc_full_PPMI, method = "delong")
delong_HBS  <- roc.test(roc_null_HBS,  roc_full_HBS,  method = "delong")

cat("DeLong test p-values:\n")
cat("  PPMI:", delong_PPMI$p.value, "\n")
cat("  HBS:",  delong_HBS$p.value,  "\n")

# -----------------------------------------------
# Extract sensitivity, specificity, accuracy at
# optimal threshold (closest-to-top-left)
# -----------------------------------------------

coords_null_PPMI <- coords(roc_null_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_PPMI <- coords(roc_full_PPMI, x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))

coords_null_HBS  <- coords(roc_null_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))
coords_full_HBS  <- coords(roc_full_HBS,  x = "best", best.method = "closest.topleft",
                           ret = c("sensitivity", "specificity", "accuracy"))


# -----------------------------------------------
# Build results table
# -----------------------------------------------

results <- rbindlist(list(
  data.table(
    cohort         = "PPMI",
    AUC_null       = as.numeric(auc(roc_null_PPMI)),
    AUC_lower_null = as.numeric(ci_null_PPMI[1]),
    AUC_upper_null = as.numeric(ci_null_PPMI[3]),
    AUC_full       = as.numeric(auc(roc_full_PPMI)),
    AUC_lower_full = as.numeric(ci_full_PPMI[1]),
    AUC_upper_full = as.numeric(ci_full_PPMI[3]),
    AUC_diff       = as.numeric(auc(roc_full_PPMI)) - as.numeric(auc(roc_null_PPMI)),
    sens_null      = coords_null_PPMI$sensitivity,
    sens_full      = coords_full_PPMI$sensitivity,
    sens_diff      = coords_full_PPMI$sensitivity - coords_null_PPMI$sensitivity,
    spec_null      = coords_null_PPMI$specificity,
    spec_full      = coords_full_PPMI$specificity,
    spec_diff      = coords_full_PPMI$specificity - coords_null_PPMI$specificity,
    acc_null       = coords_null_PPMI$accuracy,
    acc_full       = coords_full_PPMI$accuracy,
    acc_diff       = coords_full_PPMI$accuracy - coords_null_PPMI$accuracy,
    delong_z       = delong_PPMI$statistic,
    delong_p       = delong_PPMI$p.value
  ),
  data.table(
    cohort         = "HBS",
    AUC_null       = as.numeric(auc(roc_null_HBS)),
    AUC_lower_null = as.numeric(ci_null_HBS[1]),
    AUC_upper_null = as.numeric(ci_null_HBS[3]),
    AUC_full       = as.numeric(auc(roc_full_HBS)),
    AUC_lower_full = as.numeric(ci_full_HBS[1]),
    AUC_upper_full = as.numeric(ci_full_HBS[3]),
    AUC_diff       = as.numeric(auc(roc_full_HBS)) - as.numeric(auc(roc_null_HBS)),
    sens_null      = coords_null_HBS$sensitivity,
    sens_full      = coords_full_HBS$sensitivity,
    sens_diff      = coords_full_HBS$sensitivity - coords_null_HBS$sensitivity,
    spec_null      = coords_null_HBS$specificity,
    spec_full      = coords_full_HBS$specificity,
    spec_diff      = coords_full_HBS$specificity - coords_null_HBS$specificity,
    acc_null       = coords_null_HBS$accuracy,
    acc_full       = coords_full_HBS$accuracy,
    acc_diff       = coords_full_HBS$accuracy - coords_null_HBS$accuracy,
    delong_z       = delong_HBS$statistic,
    delong_p       = delong_HBS$p.value
  )
))

print(results)

results_outfile <- "/home/jupyter/multiTRS/results/multi_TRS_all_scores_ENET_external_validation_results_table.txt"
write.table(results, results_outfile, sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
cat("Results table saved to:", results_outfile, "\n")

### Transfer results across

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/*_nestedcv.txt {WORKSPACE_BUCKET}')
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/*_external_validation_results_table.txt {WORKSPACE_BUCKET}')